In [1]:
# Imports
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nested_sampler as ns

from hysys import open_case
from helpers import Surrogate
import hysys_methanol as loop          # original flowsheet, PFR solved by HYSYS
import hysys_surrogate as sur          # cut flowsheet, PFR replaced by the net

In [2]:
# KNOBS
SMOKE_TEST = False                 # few points to check the machinery; False for the real run

CASES = Path("C:/Users/kr326/Documents/Andrea_Files_MeOH_Aspen/TestAspenFile")
CASE_TRUE = str(CASES / "methanol.hsc")
CASE_SUR = str(CASES / "open_methanol.hsc")
NET = "raw"                        # "raw" or "stoi", see PFR_Explore
VOLUME = 35                        # m3, the volume both nets were trained at

# design box; key order defines the sampler's x
design_bounds = {"temperature_C": [220, 280], "pressure_bar": [70, 120]}
RATIO, PURGE_RATE = 3.0, 0.03      # fixed

# constraints, feasible iff residual <= 0 (see constraints below)
MEOH_MIN = 18_000                  # kg/h methanol product
COMPRESSION_MAX = 4_000            # kW, syngas + recycle compressors
CO_MAX = 6                         # kgmole/h net CO formed in the reactor

N_LIVE, N_PROP = (8, 2) if SMOKE_TEST else (200, 20)
MAX_TIME = 300 if SMOKE_TEST else 7200      # s wall time per sampler run

In [3]:
# Setup: both cases must be open in HYSYS (or they are launched)
objects_true = loop.cache_objects(open_case(CASE_TRUE))
objects_sur = sur.cache_objects(open_case(CASE_SUR))
net = Surrogate.load(f"pfr_surrogate_{NET}.txt")
if NET == "stoi":
    net = sur.stoi_as_raw(net)
print("cases open, net loaded:", NET)

cases open, net loaded: raw


In [4]:
# Interface adapters: design vector -> one result row, one per flowsheet
def row_hysys(x):
    T, P = x
    return loop.run_point(objects_true, pressure=(P, "bar"), temperature=(T, "C"),
                          volume=(VOLUME, "m3"), ratio=RATIO, purge_rate=PURGE_RATE)

def row_surrogate(x):
    T, P = x
    return sur.run_point(objects_sur, net, pressure=(P, "bar"), temperature=(T, "C"),
                         ratio=RATIO, purge_rate=PURGE_RATE)

# Constraints on the row, feasible iff every residual <= 0
def constraints(row):
    if not row["converged"]:
        raise RuntimeError("not converged")          # sampler drops the point, counts a failure
    return [MEOH_MIN - row["methanol_kg_h"],
            row["compression_kW"] - COMPRESSION_MAX,
            row["co_formation_kgmole_h"] - CO_MAX]

In [5]:
def nested_sample_flowsheet(box, model, constraints, *, n_live, n_prop, max_time=0):
    """Nested-sample the region where constraints(model(x)) <= 0 over `box`
    ({name: [lb, ub]}, order defines x).  Returns the sampler Result and a
    DataFrame with one row per evaluation, failed ones included."""
    rows = []

    def g(x):
        row = {**dict(zip(box, x)), **model(x)}
        rows.append(row)                          # kept even if the constraints raise
        residuals = constraints(row)
        row["max_residual"] = max(residuals)      # <= 0 means feasible
        return residuals

    lb, ub = zip(*box.values())
    result = ns.NestedSampler(g, lb, ub, num_live=n_live, num_prop=n_prop, max_time=max_time).sample()
    return result, pd.DataFrame(rows)

In [6]:
runs = {}
for name, model in (("surrogate", row_surrogate), ("hysys", row_hysys)):
    t0 = time.time()
    result, df = nested_sample_flowsheet(design_bounds, model, constraints,
                                         n_live=N_LIVE, n_prop=N_PROP, max_time=MAX_TIME)
    df["feasible"] = df["max_residual"] <= 0
    runs[name] = dict(result=result, df=df, wall_s=time.time() - t0)
    print(f"=== {name}: {len(df)} evaluations, {int(df.feasible.sum())} feasible, "
          f"{result.stats.failures} failed, status {result.status.name}, {runs[name]['wall_s']:.0f} s")


** INITIALIZING LIVE POINTS (8)      5.73 SEC

Iterate        Contour #Feas    #Dead         Factor
----------------------------------------------------
      0     2.3840e+03     0        0     3.0000e-01
     19    -1.1061e-01     9       20     1.8657e-01
=== surrogate: 46 evaluations, 9 feasible, 0 failed, status NORMAL, 24 s

** INITIALIZING LIVE POINTS (8)     17.03 SEC

Iterate        Contour #Feas    #Dead         Factor
----------------------------------------------------
      0     2.3629e+03     0        0     3.0000e-01
     16    -5.8825e-02     8       17     2.0110e-01
=== hysys: 40 evaluations, 8 feasible, 0 failed, status NORMAL, 58 s


In [ ]:
# Verification: every point the surrogate called feasible is re-evaluated on HYSYS
sur_df, hys_df = runs["surrogate"]["df"], runs["hysys"]["df"]
cand = sur_df[sur_df.feasible].copy()
t0 = time.time()
checks = [row_hysys((r.temperature_C, r.pressure_bar)) for r in cand.itertuples()]
cand["hysys_max_residual"] = [max(constraints(c)) if c["converged"] else np.nan for c in checks]
cand["verified"] = cand.hysys_max_residual <= 0
print(f"{len(cand)} surrogate-feasible points re-run on HYSYS in {time.time() - t0:.0f} s: "
      f"{int(cand.verified.sum())} confirmed, {int((~cand.verified).sum())} rejected")
print(cand[["temperature_C", "pressure_bar", "max_residual", "hysys_max_residual", "verified"]].round(2).to_string(index=False))

In [ ]:
# Surrogate feasibility against the independent HYSYS benchmark
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(sur_df.temperature_C[~sur_df.feasible], sur_df.pressure_bar[~sur_df.feasible],
           s=10, color="0.85", label="surrogate: infeasible")
ax.scatter(cand.temperature_C, cand.pressure_bar, s=40, facecolors="none", edgecolors="C0", lw=1.2,
           label="surrogate: feasible")
ax.scatter(cand.temperature_C[cand.verified], cand.pressure_bar[cand.verified], s=12, color="C0",
           label="... confirmed on HYSYS")
ax.scatter(cand.temperature_C[~cand.verified], cand.pressure_bar[~cand.verified], s=40, marker="x", color="C3",
           label="... rejected by HYSYS")
ax.scatter(hys_df.temperature_C[hys_df.feasible], hys_df.pressure_bar[hys_df.feasible], s=40, marker="+", color="k",
           label="HYSYS benchmark: feasible")
ax.set(xlabel="T [°C]", ylabel="P [bar]",
       title=f"surrogate {len(sur_df)} evals / {runs['surrogate']['wall_s']:.0f} s, "
             f"HYSYS {len(hys_df)} evals / {runs['hysys']['wall_s']:.0f} s")
ax.legend(loc="lower left", fontsize=8)
fig.tight_layout()